In [2]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape

df = pd.read_csv(
    "data/pavement-condition-rating.csv",
    sep=";",              # <-- critical
    engine="python"
)

df.columns

FileNotFoundError: [Errno 2] No such file or directory: 'data/pavement-condition-rating.csv'

In [ ]:
df["geometry"] = df["Geom"].apply(
    lambda g: shape(json.loads(g)) if isinstance(g, str) else shape(g)
)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

In [ ]:
gdf.geometry.geom_type.value_counts()

LineString         9013
MultiLineString       3
Name: count, dtype: int64

In [ ]:
from shapely.geometry import LineString, MultiLineString

def to_single_line(geom):
    if isinstance(geom, MultiLineString):
        return max(list(geom.geoms), key=lambda g: g.length)
    return geom

gdf["geometry"] = gdf["geometry"].apply(to_single_line)

In [ ]:
gdf = gdf.to_crs(epsg=26910)  # Vancouver UTM zone

In [ ]:
gdf["segment_length_m"] = gdf.geometry.length

In [ ]:
import numpy as np

def bearing_deg(line):
    (x1, y1), (x2, y2) = list(line.coords)[0], list(line.coords)[-1]
    ang = np.degrees(np.arctan2(x2 - x1, y2 - y1))
    return (ang + 360) % 360

gdf["bearing_deg"] = gdf.geometry.apply(bearing_deg)

In [ ]:
def sun_exposure_score(b):
    if 135 <= b <= 225:       # south-facing
        return 0.2
    elif b <= 45 or b >= 315: # north-facing
        return 0.8
    else:
        return 0.5

gdf["sun_exposure"] = gdf["bearing_deg"].apply(sun_exposure_score)

In [ ]:
gdf

,Year,Road Name,From Street,To Street,length_(m),PCI Rating,Geom,geo_point_2d,geometry,segment_length_m,bearing_deg,sun_exposure
0,2020,47TH AV,FRONTENAC ST,KIRKLAND ST,89,GOOD,"{""coordinates"": [[-123.02596132078351, 49.2263...","49.22637498558873, -123.02535015595373","LINESTRING (498109.744 5452621.933, 498198.525...",89.000410,90.403261,0.5
1,2020,62ND AV,LABURNUM ST,ANGUS DRIVE,206,VERY POOR,"{""coordinates"": [[-123.15124781549908, 49.2151...","49.215083474080224, -123.1498338331115","LINESTRING (488985.072 5451379.255, 489134.638...",206.000075,91.342846,0.5
2,2020,29TH AV,ELGIN ST,ROSS ST,101,POOR,"{""coordinates"": [[-123.08272270544481, 49.2446...","49.244605930447584, -123.08202889157728","LINESTRING (493979.138 5454651.721, 494029.134...",100.999671,90.501658,0.5
3,2020,31ST AV,LANARK ST,DUMFRIES ST,93,VERY POOR,"{""coordinates"": [[-123.07483046938307, 49.2430...","49.243003606888095, -123.07419187431712","LINESTRING (494553.388 5454473.965, 494596.652...",92.999780,91.658554,0.5
4,2020,53RD AV,KNIGHT ST,LANARK ST,100,VERY GOOD,"{""coordinates"": [[-123.07741967350125, 49.2217...","49.22178989465209, -123.07673312590855","LINESTRING (494362.517 5452115.319, 494412.381...",99.999508,91.037803,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...
9011,2020,14TH AV,BLANCA ST,TOLMIE ST,224,VERY GOOD,"{""coordinates"": [[-123.21534324346365, 49.2602...","49.2602645177521, -123.21380439383668","LINESTRING (484331.496 5456413.79, 484555.123 ...",223.999862,91.529059,0.5
9012,2020,BLENHEIM ST,27TH AV,26TH AV,105,VERY GOOD,"{""coordinates"": [[-123.17860520439913, 49.2480...","49.248519781794535, -123.17860064822496","LINESTRING (487001.362 5455045.993, 487001.82 ...",104.999880,0.492477,0.8
9013,2020,51ST AV,PRINCE EDWARD ST,ST. GEORGE ST,250,VERY GOOD,"{""coordinates"": [[-123.09756290355297, 49.2236...","49.22365170028139, -123.0958468014427","LINESTRING (492896.013 5452326.414, 493086.788...",250.000105,91.575442,0.5
9014,2020,BALACLAVA ST,28TH AV,27TH AV,106,VERY GOOD,"{""coordinates"": [[-123.1757109133116, 49.24707...","49.24755564087381, -123.17570679053291","LINESTRING (487211.755 5454937.815, 487211.757...",106.000175,0.457235,0.8


In [ ]:
pci_map = {
    "Excellent": 0.1,
    "Good": 0.3,
    "Fair": 0.5,
    "Poor": 0.8,
    "Very Poor": 0.95
}
gdf["pavement_risk"] = gdf["PCI Rating"].map(pci_map)

In [ ]:
import numpy as np

# Ensure geometry exists and is valid-ish
gdf = gdf[gdf.geometry.notna()].copy()
gdf = gdf[~gdf.geometry.is_empty].copy()

# Fix invalid geometries (cheap fix)
gdf["geometry"] = gdf.geometry.buffer(0)

In [ ]:
import numpy as np

print("Empty geometries:", gdf.geometry.is_empty.sum(), "out of", len(gdf))
print("NaN lengths:", gdf.geometry.length.isna().sum())
print("Zero lengths:", (gdf.geometry.length == 0).sum())

Empty geometries: 9016 out of 9016
NaN lengths: 0
Zero lengths: 9016


In [ ]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString

def geojson_to_linestring(g):
    # g can be a string or dict
    if pd.isna(g):
        return None
    if isinstance(g, str):
        g = json.loads(g)

    if not isinstance(g, dict):
        return None
    if g.get("type") not in ("LineString", "MultiLineString"):
        return None

    coords = g.get("coordinates", None)
    if not coords:
        return None

    # If MultiLineString, take the longest part
    if g["type"] == "MultiLineString":
        lines = [LineString(part) for part in coords if part and len(part) >= 2]
        if not lines:
            return None
        return max(lines, key=lambda ln: ln.length)

    # LineString
    if len(coords) < 2:
        return None
    return LineString(coords)

df["geometry"] = df["Geom"].apply(geojson_to_linestring)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

In [ ]:
gdf = gdf.dropna(subset=["geometry"]).copy()
gdf = gdf[~gdf.geometry.is_empty].copy()

print("Remaining rows:", len(gdf))
print(gdf.geometry.geom_type.value_counts())
print("Any empty left?", gdf.geometry.is_empty.any())
print("Example bounds:", gdf.geometry.iloc[0].bounds)

Remaining rows: 9016
LineString    9016
Name: count, dtype: int64
Any empty left? False
Example bounds: (-123.02596132078351, 49.226372308631696, -123.02473899109518, 49.22637767566639)


In [ ]:
gdf_ll = gdf  # already EPSG:4326
minx, miny, maxx, maxy = gdf_ll.total_bounds
print("Bounds:", minx, miny, maxx, maxy)

Bounds: -123.22394600626724 49.20032273588974 -123.02294302637398 49.3125375062427


In [ ]:
import osmnx as ox

gdf = gdf.to_crs(epsg=26910)

tags = {"natural": "water"}
water = ox.features_from_place("Vancouver, British Columbia, Canada", tags=tags)
water = water[water.geometry.type.isin(["Polygon","MultiPolygon"])].to_crs(gdf.crs)

water_union = water.unary_union
gdf["dist_to_water_m"] = gdf.geometry.apply(lambda g: g.distance(water_union))

import numpy as np
k = 500
gdf["humidity_proxy"] = np.exp(-gdf["dist_to_water_m"] / k)

/var/folders/wg/l91gh0qs7wsdbpqj6jhpsdfh0000gn/T/ipykernel_5320/1595943033.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  water_union = water.unary_union


In [ ]:
gdf

,Year,Road Name,From Street,To Street,length_(m),PCI Rating,Geom,geo_point_2d,geometry,dist_to_water_m,humidity_proxy
0,2020,47TH AV,FRONTENAC ST,KIRKLAND ST,89,GOOD,"{""coordinates"": [[-123.02596132078351, 49.2263...","49.22637498558873, -123.02535015595373","LINESTRING (498109.744 5452621.933, 498198.525...",1601.867120,0.040610
1,2020,62ND AV,LABURNUM ST,ANGUS DRIVE,206,VERY POOR,"{""coordinates"": [[-123.15124781549908, 49.2151...","49.215083474080224, -123.1498338331115","LINESTRING (488985.072 5451379.255, 489134.638...",549.102888,0.333469
2,2020,29TH AV,ELGIN ST,ROSS ST,101,POOR,"{""coordinates"": [[-123.08272270544481, 49.2446...","49.244605930447584, -123.08202889157728","LINESTRING (493979.138 5454651.721, 494029.134...",1763.966416,0.029366
3,2020,31ST AV,LANARK ST,DUMFRIES ST,93,VERY POOR,"{""coordinates"": [[-123.07483046938307, 49.2430...","49.243003606888095, -123.07419187431712","LINESTRING (494553.388 5454473.965, 494596.652...",1512.928922,0.048516
4,2020,53RD AV,KNIGHT ST,LANARK ST,100,VERY GOOD,"{""coordinates"": [[-123.07741967350125, 49.2217...","49.22178989465209, -123.07673312590855","LINESTRING (494362.517 5452115.319, 494412.381...",1435.565767,0.056635
...,...,...,...,...,...,...,...,...,...,...,...
9011,2020,14TH AV,BLANCA ST,TOLMIE ST,224,VERY GOOD,"{""coordinates"": [[-123.21534324346365, 49.2602...","49.2602645177521, -123.21380439383668","LINESTRING (484331.496 5456413.79, 484555.123 ...",1570.808455,0.043213
9012,2020,BLENHEIM ST,27TH AV,26TH AV,105,VERY GOOD,"{""coordinates"": [[-123.17860520439913, 49.2480...","49.248519781794535, -123.17860064822496","LINESTRING (487001.362 5455045.993, 487001.82 ...",2238.463814,0.011368
9013,2020,51ST AV,PRINCE EDWARD ST,ST. GEORGE ST,250,VERY GOOD,"{""coordinates"": [[-123.09756290355297, 49.2236...","49.22365170028139, -123.0958468014427","LINESTRING (492896.013 5452326.414, 493086.788...",662.208153,0.265958
9014,2020,BALACLAVA ST,28TH AV,27TH AV,106,VERY GOOD,"{""coordinates"": [[-123.1757109133116, 49.24707...","49.24755564087381, -123.17570679053291","LINESTRING (487211.755 5454937.815, 487211.757...",2152.083315,0.013512


In [ ]:
pci_map = {
    "VERY GOOD": 0.1,
    "GOOD": 0.3,
    "FAIR": 0.5,
    "POOR": 0.8,
    "VERY POOR": 0.95
}

gdf["pavement_risk"] = gdf["PCI Rating"].map(pci_map)

In [ ]:
gdf["pci_missing"] = gdf["pavement_risk"].isna().astype(int)

In [ ]:
default_pci = gdf["pavement_risk"].median()  # usually ~0.5
gdf["pavement_risk"] = gdf["pavement_risk"].fillna(default_pci)

In [ ]:
gdf["pavement_risk_adj"] = gdf["pavement_risk"] * (1 - 0.2 * gdf["pci_missing"])

In [ ]:
import numpy as np

def bearing_deg(line):
    (x1, y1), (x2, y2) = list(line.coords)[0], list(line.coords)[-1]
    ang = np.degrees(np.arctan2(x2 - x1, y2 - y1))
    return (ang + 360) % 360

gdf["bearing_deg"] = gdf.geometry.apply(bearing_deg)

def sun_exposure_score(b):
    # Simple heuristic: north-facing = less sun = higher ice risk
    if 135 <= b <= 225:        # south-ish
        return 0.2
    elif b <= 45 or b >= 315:  # north-ish
        return 0.8
    else:
        return 0.5

gdf["sun_exposure"] = gdf["bearing_deg"].apply(sun_exposure_score)

In [ ]:
import osmnx as ox
import geopandas as gpd

tags_bridge = {"bridge": "yes"}
bridges = ox.features_from_place("Vancouver, British Columbia, Canada", tags=tags_bridge)

# keep linear features
bridges = bridges[bridges.geometry.type.isin(["LineString","MultiLineString"])].copy()
bridges = bridges.to_crs(gdf.crs)

In [ ]:
bridges_union = bridges.unary_union
gdf["is_bridge"] = gdf.geometry.apply(lambda g: g.intersects(bridges_union)).astype(int)

/var/folders/wg/l91gh0qs7wsdbpqj6jhpsdfh0000gn/T/ipykernel_5320/2137807819.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  bridges_union = bridges.unary_union


In [ ]:
T = 0.5   # degrees C (example)
P6 = 2.0  # mm precip last 6 hours (example)

gdf["recent_moisture"] = int(P6 > 0)
gdf["temp_near_zero"] = max(0, 1 - abs(T)/5)   # peaks at 0C, fades by +-5C

In [ ]:
gdf.columns

Index(['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)',
       'PCI Rating', 'Geom', 'geo_point_2d', 'geometry', 'dist_to_water_m',
       'humidity_proxy', 'pavement_risk', 'pci_missing', 'pavement_risk_adj',
       'bearing_deg', 'sun_exposure', 'is_bridge', 'recent_moisture',
       'temp_near_zero'],
      dtype='object')

In [ ]:
keep_cols = [
    "year", "road_name", "from_street", "to_street",
    "segment_length_m", "pci_rating",
    "dist_to_water_m", "humidity_proxy",
    "pavement_risk", "pci_missing", "pavement_risk_adj",
    "bearing_deg", "sun_exposure", "is_bridge",
    "recent_moisture", "temp_near_zero",
    "geometry"
]
# Some of these might not exist yet depending on your progress; keep what exists
keep_cols = [c for c in keep_cols if c in gdf.columns]
gdf_backup = gdf.copy()
gdf = gdf[keep_cols].copy()

gdf.head()

,dist_to_water_m,humidity_proxy,pavement_risk,pci_missing,pavement_risk_adj,bearing_deg,sun_exposure,is_bridge,recent_moisture,temp_near_zero,geometry
0,1601.867120,0.040610,0.30,0,0.30,90.403261,0.5,0,1,0.9,"LINESTRING (498109.744 5452621.933, 498198.525..."
1,549.102888,0.333469,0.95,0,0.95,91.342846,0.5,0,1,0.9,"LINESTRING (488985.072 5451379.255, 489134.638..."
2,1763.966416,0.029366,0.80,0,0.80,90.501658,0.5,0,1,0.9,"LINESTRING (493979.138 5454651.721, 494029.134..."
3,1512.928922,0.048516,0.95,0,0.95,91.658554,0.5,0,1,0.9,"LINESTRING (494553.388 5454473.965, 494596.652..."
4,1435.565767,0.056635,0.10,0,0.10,91.037803,0.5,0,1,0.9,"LINESTRING (494362.517 5452115.319, 494412.381..."


In [ ]:
# Columns that should be numeric
numeric_cols = [
    "segment_length_m", "dist_to_water_m", "humidity_proxy",
    "pavement_risk", "pci_missing", "pavement_risk_adj",
    "bearing_deg", "sun_exposure", "is_bridge",
    "recent_moisture", "temp_near_zero"
]
numeric_cols = [c for c in numeric_cols if c in gdf.columns]

# Convert to numeric safely
for c in numeric_cols:
    gdf[c] = pd.to_numeric(gdf[c], errors="coerce")

# Replace +/-inf with NaN
gdf[numeric_cols] = gdf[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Fill missingness appropriately
# - humidity_proxy: if missing, assume not near water (low humidity effect)
if "humidity_proxy" in gdf.columns:
    gdf["humidity_proxy"] = gdf["humidity_proxy"].fillna(0.0)

# - dist_to_water_m: if missing, set large distance (conservative)
if "dist_to_water_m" in gdf.columns:
    gdf["dist_to_water_m"] = gdf["dist_to_water_m"].fillna(gdf["dist_to_water_m"].median())

# - sun_exposure: neutral if missing
if "sun_exposure" in gdf.columns:
    gdf["sun_exposure"] = gdf["sun_exposure"].fillna(0.5)

# - is_bridge: missing => not bridge
if "is_bridge" in gdf.columns:
    gdf["is_bridge"] = gdf["is_bridge"].fillna(0).astype(int)

# - recent_moisture: missing => 0
if "recent_moisture" in gdf.columns:
    gdf["recent_moisture"] = gdf["recent_moisture"].fillna(0).astype(int)

# - temp_near_zero: missing => 0 (no risk contribution)
if "temp_near_zero" in gdf.columns:
    gdf["temp_near_zero"] = gdf["temp_near_zero"].fillna(0.0)

# - pavement_risk_adj: if missing, fall back to pavement_risk; else median; plus flag pci_missing
if "pavement_risk_adj" in gdf.columns:
    if "pavement_risk" in gdf.columns:
        gdf["pavement_risk_adj"] = gdf["pavement_risk_adj"].fillna(gdf["pavement_risk"])
    gdf["pavement_risk_adj"] = gdf["pavement_risk_adj"].fillna(gdf["pavement_risk_adj"].median())

if "pci_missing" in gdf.columns:
    gdf["pci_missing"] = gdf["pci_missing"].fillna(1).astype(int)

# Optional: drop any rows that still have no geometry
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

gdf.describe(include="all")

,dist_to_water_m,humidity_proxy,pavement_risk,pci_missing,pavement_risk_adj,bearing_deg,sun_exposure,is_bridge,recent_moisture,temp_near_zero,geometry
count,9016.000000,9016.000000,9016.000000,9016.000000,9016.000000,9016.000000,9016.000000,9016.000000,9016.0,9.016000e+03,9016
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9016
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LINESTRING (498109.74409996445 5452621.9332963...
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
mean,894.895877,0.269265,0.510210,0.047693,0.505440,69.482194,0.642946,0.010426,1.0,9.000000e-01,NaN
std,548.670870,0.235776,0.301095,0.213128,0.302009,81.284714,0.157000,0.101579,0.0,2.220569e-16,NaN
min,0.000000,0.005071,0.100000,0.000000,0.100000,0.000299,0.200000,0.000000,1.0,9.000000e-01,NaN
25%,464.558417,0.078719,0.300000,0.000000,0.300000,1.484126,0.500000,0.000000,1.0,9.000000e-01,NaN
50%,818.423744,0.194593,0.500000,0.000000,0.500000,90.098423,0.500000,0.000000,1.0,9.000000e-01,NaN
75%,1270.932607,0.394902,0.800000,0.000000,0.800000,91.506359,0.800000,0.000000,1.0,9.000000e-01,NaN


In [ ]:
# Features to normalize (continuous)
norm_cols = []
for c in ["humidity_proxy", "pavement_risk_adj", "sun_exposure", "dist_to_water_m"]:
    if c in gdf.columns:
        norm_cols.append(c)

# Binary / already-scaled columns (don't normalize)
binary_cols = [c for c in ["is_bridge", "recent_moisture", "pci_missing"] if c in gdf.columns]

# Sanity checks
print("Rows:", len(gdf))
print("CRS:", gdf.crs)
print("Normalize:", norm_cols)
print("Binary:", binary_cols)

# Ensure temp_near_zero is within [0,1]
if "temp_near_zero" in gdf.columns:
    gdf["temp_near_zero"] = gdf["temp_near_zero"].clip(0, 1)

# Ensure humidity_proxy is within [0,1]
if "humidity_proxy" in gdf.columns:
    gdf["humidity_proxy"] = gdf["humidity_proxy"].clip(0, 1)

gdf.head()

Rows: 9016
CRS: EPSG:26910
Normalize: ['humidity_proxy', 'pavement_risk_adj', 'sun_exposure', 'dist_to_water_m']
Binary: ['is_bridge', 'recent_moisture', 'pci_missing']


,dist_to_water_m,humidity_proxy,pavement_risk,pci_missing,pavement_risk_adj,bearing_deg,sun_exposure,is_bridge,recent_moisture,temp_near_zero,geometry
0,1601.867120,0.040610,0.30,0,0.30,90.403261,0.5,0,1,0.9,"LINESTRING (498109.744 5452621.933, 498198.525..."
1,549.102888,0.333469,0.95,0,0.95,91.342846,0.5,0,1,0.9,"LINESTRING (488985.072 5451379.255, 489134.638..."
2,1763.966416,0.029366,0.80,0,0.80,90.501658,0.5,0,1,0.9,"LINESTRING (493979.138 5454651.721, 494029.134..."
3,1512.928922,0.048516,0.95,0,0.95,91.658554,0.5,0,1,0.9,"LINESTRING (494553.388 5454473.965, 494596.652..."
4,1435.565767,0.056635,0.10,0,0.10,91.037803,0.5,0,1,0.9,"LINESTRING (494362.517 5452115.319, 494412.381..."


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
gdf[norm_cols] = scaler.fit_transform(gdf[norm_cols])

In [ ]:
gdf["black_ice_risk"] = (
    0.35 * gdf["temp_near_zero"] +
    0.25 * gdf["recent_moisture"] +
    0.15 * gdf["humidity_proxy"] +
    0.15 * gdf["pavement_risk"] +
    0.10 * gdf["sun_exposure"] +
    0.20 * gdf["is_bridge"]   # bridges matter a lot; ok to overweight
)

# Optional: clip to [0,1]
gdf["black_ice_risk"] = gdf["black_ice_risk"].clip(0, 1)

In [ ]:
gdf

,dist_to_water_m,humidity_proxy,pavement_risk,pci_missing,pavement_risk_adj,bearing_deg,sun_exposure,is_bridge,recent_moisture,temp_near_zero,geometry,black_ice_risk
0,0.606275,0.035721,0.30,0,0.235294,90.403261,0.5,0,1,0.9,"LINESTRING (498109.744 5452621.933, 498198.525...",0.665358
1,0.207825,0.330072,0.95,0,1.000000,91.342846,0.5,0,1,0.9,"LINESTRING (488985.072 5451379.255, 489134.638...",0.807011
2,0.667627,0.024419,0.80,0,0.823529,90.501658,0.5,0,1,0.9,"LINESTRING (493979.138 5454651.721, 494029.134...",0.738663
3,0.572614,0.043667,0.95,0,1.000000,91.658554,0.5,0,1,0.9,"LINESTRING (494553.388 5454473.965, 494596.652...",0.764050
4,0.543334,0.051827,0.10,0,0.000000,91.037803,0.5,0,1,0.9,"LINESTRING (494362.517 5452115.319, 494412.381...",0.637774
...,...,...,...,...,...,...,...,...,...,...,...,...
9011,0.594520,0.038337,0.10,0,0.000000,91.529059,0.5,0,1,0.9,"LINESTRING (484331.496 5456413.79, 484555.123 ...",0.635750
9012,0.847215,0.006330,0.10,0,0.000000,0.492477,1.0,0,1,0.9,"LINESTRING (487001.362 5455045.993, 487001.82 ...",0.680949
9013,0.250633,0.262217,0.10,0,0.000000,91.575442,0.5,0,1,0.9,"LINESTRING (492896.013 5452326.414, 493086.788...",0.669333
9014,0.814521,0.008485,0.10,0,0.000000,0.457235,1.0,0,1,0.9,"LINESTRING (487211.755 5454937.815, 487211.757...",0.681273


In [ ]:
top = gdf.sort_values("black_ice_risk", ascending=False).head(20)
top[[
    "black_ice_risk",
    "is_bridge",
    "dist_to_water_m",
    "humidity_proxy",
    "sun_exposure",
    "pavement_risk_adj",
    "recent_moisture",
    "temp_near_zero"
]]

,black_ice_risk,is_bridge,dist_to_water_m,humidity_proxy,sun_exposure,pavement_risk_adj,recent_moisture,temp_near_zero
6683,1.0,1,0.025928,0.871307,0.5,0.470588,1,0.9
4089,1.0,1,0.007356,0.961678,0.5,1.000000,1,0.9
8904,1.0,1,0.025425,0.873641,0.5,0.470588,1,0.9
2395,1.0,1,0.068665,0.694143,0.5,0.823529,1,0.9
5631,1.0,1,0.079791,0.654217,1.0,0.823529,1,0.9
8794,1.0,1,0.100444,0.586049,1.0,0.823529,1,0.9
6323,1.0,1,0.027439,0.864336,0.5,0.470588,1,0.9
6438,1.0,1,0.109669,0.557924,0.5,0.823529,1,0.9
8524,1.0,1,0.012938,0.933580,0.5,0.823529,1,0.9
378,1.0,1,0.048861,0.771286,1.0,0.470588,1,0.9


In [ ]:
import pandas as pd

src = pd.read_csv("data/pavement-condition-rating.csv", sep=";", engine="python")
src = src.reset_index(drop=True)
src["segment_id"] = src.index

In [ ]:
gdf = gdf.reset_index(drop=True)
gdf["segment_id"] = gdf.index

gdf = gdf.merge(
    src[["segment_id", "Road Name", "From Street", "To Street", "PCI Rating", "Year"]],
    on="segment_id",
    how="left"
)

gdf = gdf.rename(columns={
    "Road Name": "road_name",
    "From Street": "from_street",
    "To Street": "to_street",
    "PCI Rating": "pci_rating",
    "Year": "year",
})

In [ ]:
top = gdf.sort_values("black_ice_risk", ascending=False).head(20)
top[["road_name","from_street","to_street","black_ice_risk","is_bridge","dist_to_water_m","pci_rating"]]

,road_name,from_street,to_street,black_ice_risk,is_bridge,dist_to_water_m,pci_rating
6683,SMITHE ST,HOWE ST,HORNBY ST,1.0,1,0.025928,FAIR
4089,EXPO BLVD,NELSON ST,CAMBIE ST,1.0,1,0.007356,VERY POOR
8904,12TH AV,LAUREL ST,WILLOW ST,1.0,1,0.025425,FAIR
2395,75TH AV,HUDSON ST,EAST END,1.0,1,0.068665,POOR
5631,FIR ST,4TH AV,3RD AV,1.0,1,0.079791,POOR
8794,QUEBEC ST,TERMINAL AV,NATIONAL AV,1.0,1,0.100444,POOR
6323,6TH AV,OAK ST,LAUREL ST,1.0,1,0.027439,FAIR
6438,KENT AV N,OAK ST,SHAUGHNESSY ST,1.0,1,0.109669,POOR
8524,PACIFIC BLVD,CAMBIE ST,MARINASIDE CRESCENT / NELSON ST,1.0,1,0.012938,POOR
378,GRANVILLE ST 2,4TH AV,3RD AV / ANDERSON ST,1.0,1,0.048861,FAIR


In [ ]:
def top_drivers(row):
    drivers = []
    if row.get("is_bridge", 0) == 1:
        drivers.append("bridge deck")
    if row.get("humidity_proxy", 0) > 0.7:
        drivers.append("near water / high humidity")
    if row.get("sun_exposure", 0) > 0.7:
        drivers.append("low winter sun exposure")
    if row.get("recent_moisture", 0) == 1:
        drivers.append("recent precipitation")
    if row.get("pavement_risk_adj", 0) > 0.7:
        drivers.append("poor pavement condition")
    return ", ".join(drivers[:3]) if drivers else "baseline conditions"

gdf["risk_drivers"] = gdf.apply(top_drivers, axis=1)

In [ ]:
top = gdf.sort_values("black_ice_risk", ascending=False).head(20)
top[["road_name","from_street","to_street","black_ice_risk","risk_drivers"]]

,road_name,from_street,to_street,black_ice_risk,risk_drivers
6683,SMITHE ST,HOWE ST,HORNBY ST,1.0,"bridge deck, near water / high humidity, recen..."
4089,EXPO BLVD,NELSON ST,CAMBIE ST,1.0,"bridge deck, near water / high humidity, recen..."
8904,12TH AV,LAUREL ST,WILLOW ST,1.0,"bridge deck, near water / high humidity, recen..."
2395,75TH AV,HUDSON ST,EAST END,1.0,"bridge deck, recent precipitation, poor paveme..."
5631,FIR ST,4TH AV,3RD AV,1.0,"bridge deck, low winter sun exposure, recent p..."
8794,QUEBEC ST,TERMINAL AV,NATIONAL AV,1.0,"bridge deck, low winter sun exposure, recent p..."
6323,6TH AV,OAK ST,LAUREL ST,1.0,"bridge deck, near water / high humidity, recen..."
6438,KENT AV N,OAK ST,SHAUGHNESSY ST,1.0,"bridge deck, recent precipitation, poor paveme..."
8524,PACIFIC BLVD,CAMBIE ST,MARINASIDE CRESCENT / NELSON ST,1.0,"bridge deck, near water / high humidity, recen..."
378,GRANVILLE ST 2,4TH AV,3RD AV / ANDERSON ST,1.0,"bridge deck, near water / high humidity, low w..."


In [ ]:
import folium

gdf_ll = gdf.to_crs(epsg=4326)

m = folium.Map(location=[49.2827, -123.1207], zoom_start=12)

# sample for speed if needed
sample = gdf_ll.sample(min(4000, len(gdf_ll)), random_state=0)

def style_fn(feat):
    r = feat["properties"]["black_ice_risk"]
    if r >= 0.75:
        color = "#d7191c"
    elif r >= 0.5:
        color = "#fdae61"
    else:
        color = "#1a9641"
    return {"color": color, "weight": 3, "opacity": 0.8}

tooltip_fields = [c for c in ["road_name","from_street","to_street","black_ice_risk","risk_drivers","pci_rating","is_bridge"] if c in sample.columns]

folium.GeoJson(
    sample,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields)
).add_to(m)

m

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf.crs) 

place = "Vancouver, British Columbia, Canada" 
G_full = ox.graph_from_place(place, network_type="walk") 
nodes_full, edges_full = ox.graph_to_gdfs(G_full)

edges_full = edges_full.to_crs(gdf.crs)

# --- Pavement midpoints (keep 'midpoint' as active geometry) ---
pav = gdf.copy()
pav["midpoint"] = pav.geometry.interpolate(0.5, normalized=True)
pav_mid = pav.set_geometry("midpoint")
pav_mid = pav_mid.set_crs(gdf.crs, allow_override=True)

# Keep only needed columns BUT keep 'midpoint' because it's the active geometry
pav_mid = pav_mid[[
    "black_ice_risk", "pavement_risk_adj", "pci_rating", "year", "risk_drivers", "midpoint"
]]

# --- Full streets midpoints ---
streets = edges_full.copy()
streets["midpoint"] = streets.geometry.interpolate(0.5, normalized=True)
streets_mid = streets.set_geometry("midpoint")
streets_mid = streets_mid.set_crs(edges_full.crs, allow_override=True)

# --- Ensure both are same CRS ---
pav_mid = pav_mid.to_crs(streets_mid.crs)

# --- Nearest join ---
enriched = gpd.sjoin_nearest(
    streets_mid,
    pav_mid,
    how="left",
    distance_col="dist_to_pavement"
)

edges_full["black_ice_risk"] = enriched["black_ice_risk"].values
edges_full["dist_to_pavement"] = enriched["dist_to_pavement"].values
edges_full["risk_drivers"] = enriched["risk_drivers"].fillna(
    "estimated from nearest pavement segment"
).values

import numpy as np

# Fill missing values with median risk
median_risk = float(np.nanmedian(edges_full["black_ice_risk"]))
edges_full["black_ice_risk"] = edges_full["black_ice_risk"].fillna(median_risk)

# Confidence based on distance to nearest pavement measurement
edges_full["risk_confidence"] = np.where(
    edges_full["dist_to_pavement"] <= 300,
    "high",
    "low"
)

alpha = 8.0  # how much to prioritize safety

for u, v, k, data in G_full.edges(keys=True, data=True):

    try:
        risk = float(edges_full.loc[(u, v, k), "black_ice_risk"])
    except:
        risk = median_risk

    length = data.get("length", 1.0)

    # Weight favors low-risk streets
    data["weight"] = length * (1 + alpha * risk)

    start_lonlat = (-123.1207, 49.2827)
end_lonlat   = (-123.1140, 49.2635)

orig = ox.nearest_nodes(G_full, X=start_lonlat[0], Y=start_lonlat[1])
dest = ox.nearest_nodes(G_full, X=end_lonlat[0], Y=end_lonlat[1])

route_nodes = ox.shortest_path(G_full, orig, dest, weight="weight")

print("Route length (nodes):", len(route_nodes))

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G_full)

# route_nodes is a list of node IDs; edges are between consecutive nodes
route_edges = edges_gdf.loc[[(u, v, 0) for u, v in zip(route_nodes[:-1], route_nodes[1:])]]
route_edges = route_edges.to_crs("EPSG:4326")

import folium

# --- Ensure both are lat/lon for Folium ---
gdf_ll = gdf.to_crs(epsg=4326)
route_edges_ll = route_edges.to_crs(epsg=4326)  # important

# Base map (center on Vancouver, or compute from route)
m = folium.Map(location=[49.2827, -123.1207], zoom_start=12)

# --- Layer 1: Citywide black ice risk (sampled) ---
sample = gdf_ll.sample(min(4000, len(gdf_ll)), random_state=0)

def style_fn(feat):
    r = feat["properties"].get("black_ice_risk", 0)
    if r >= 0.75:
        color = "#d7191c"
    elif r >= 0.5:
        color = "#fdae61"
    else:
        color = "#1a9641"
    return {"color": color, "weight": 3, "opacity": 0.6}

tooltip_fields = [c for c in
                  ["road_name","from_street","to_street","black_ice_risk","risk_drivers","pci_rating","is_bridge"]
                  if c in sample.columns]

risk_layer = folium.FeatureGroup(name="Black ice risk (sample)", show=True)

folium.GeoJson(
    sample,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields)
).add_to(risk_layer)

risk_layer.add_to(m)

# --- Layer 2: Route overlay (on top) ---
route_layer = folium.FeatureGroup(name="Safest route", show=True)

folium.GeoJson(
    route_edges_ll,
    style_function=lambda feat: {"color": "#000000", "weight": 7, "opacity": 0.95},
    tooltip=folium.GeoJsonTooltip(fields=[c for c in ["name", "black_ice_risk", "risk_confidence"] if c in route_edges_ll.columns])
).add_to(route_layer)

route_layer.add_to(m)

# Optional: fit map to route bounds
bounds = route_edges_ll.total_bounds  # [minx, miny, maxx, maxy]
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# Toggle layers
folium.LayerControl().add_to(m)

m

NameError: name 'gdf' is not defined